# Compute linkage disequilibrium and correlations between SNPs
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from hk1_r12ter_analysis.config import GENE_ID, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)
from hk1_r12ter_analysis.linkage_disequilibrium import (
    compute_pairwise_linkage_disequilibrium,
    compute_pairwise_spearman_correlation_genomics,
)

## Load REDS Index data

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"
data_type = f"Genomics-{GENE_ID}_Metadata"

input_dirpath = PROCESSED_DATA_DIR / "REDS"
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Donor genomics and ethnicity metadata

In [ ]:
input_filename = make_filename(source_type, "Donor", data_type)
df_ethnicity_genomics = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
# # Filter for genotyped SNPs only if desired
genomic_columns = df_ethnicity_genomics.loc[:, f"Genomics-{GENE_ID}"].columns.tolist()
df_ethnicity_genomics = df_ethnicity_genomics.loc[
    :, pd.IndexSlice[:, ["Ethnicity"] + genomic_columns]
].droplevel(0, axis=1)
df_ethnicity_genomics

### Compute linkage disequilibrium for each ethnicity subgroup

In [ ]:
def group_genomics_by_ethnicity(df, ethnicity):
    return (
        df[df["Ethnicity"] == ethnicity].drop("Ethnicity", axis=1)
        if ethnicity != "ALL"
        else df.drop("Ethnicity", axis=1)
    )


df_ld_matrices = {}
method = "RH"
recompute = False
return_LD_coefficients = True
include_spearman_coefficient = True

filename_args = [
    source_type,
    data_type,
    method,
    "LinkageDisequilibrium",
]
ethnicities = ["ALL"]  # + list(df_ethnicity_genomics["Ethnicity"].dropna().unique())
for ethnicity in ethnicities:
    # Filter data by ethnic group
    dosage_matrix = group_genomics_by_ethnicity(df_ethnicity_genomics, ethnicity)
    ld_matrix = compute_pairwise_linkage_disequilibrium(
        dosage_matrix, method=method, return_LD_coefficients=return_LD_coefficients
    )
    if include_spearman_coefficient:
        spearman_matrix = compute_pairwise_spearman_correlation_genomics(dosage_matrix)
        ld_matrix = spearman_matrix.merge(ld_matrix, left_index=True, right_index=True, how="left")
    if return_LD_coefficients:
        ld_matrix["r2 / D'"] = ld_matrix.loc[:, "r2"] / ld_matrix.loc[:, "D'"]
    df_ld_matrices[ethnicity] = ld_matrix
    output_filename = make_filename(*filename_args, ethnicity.replace("/", "_"))
    save_dataframe_to_file(ld_matrix, output_dirpath / output_filename, index=True)

if len(df_ld_matrices) > 1:
    df_ld_matrices = pd.concat(
        df_ld_matrices, keys=df_ld_matrices.keys(), names=["Ethnicity", "SNP1", "SNP2"]
    )
    output_filename = make_filename(*filename_args)
    save_dataframe_to_file(df_ld_matrices, output_dirpath / output_filename, index=True)